In [1]:
import pandas as pd
from rapidfuzz import process, fuzz, utils

customers = pd.read_csv("../data/processed/customers.csv")
names = pd.read_csv("../data/processed/names.csv")
names = names.dropna(subset=["name"]).reset_index(drop=True)

len(customers), len(names)

(5220, 16221)

In [2]:
THRESHOLD = 85

alerts = []
for _, c in customers.iterrows():
    matches = process.extract(c["name"], names["name"],
                              scorer=fuzz.token_sort_ratio,
                              processor=utils.default_process,
                              score_cutoff=THRESHOLD, limit=None)
    for matched_name, score, row in matches:
        alerts.append({"customer_id": c["customer_id"],
                       "customer_name": c["name"],
                       "sdn_id": names.loc[row, "id"],
                       "matched_name": matched_name,
                       "score": round(score),
                       "label": c["label"]})

alerts = (pd.DataFrame(alerts)
          .sort_values("score", ascending=False)
          .drop_duplicates(["customer_id", "sdn_id"]))
len(alerts)

249

In [3]:
print(alerts.groupby("label")["customer_id"].nunique())
caught = alerts[alerts["label"] == "true_match"]["customer_id"].nunique()
print(f"Złapani przestępcy: {caught} / 20")

label
clean           6
lookalike     200
true_match     20
Name: customer_id, dtype: int64
Złapani przestępcy: 20 / 20


In [4]:
alerts[alerts["label"] == "clean"]

,customer_id,customer_name,sdn_id,matched_name,score,label
160,C02982,Leonid Miguel,23365,"LEONE, Miguel",88,clean
260,C01919,Mohammad Almeida,20440,"ALHMIDAN, Mohamad",88,clean
328,C01359,Francisco Javier Madrid,24704,"DIAZ MADRIZ, Francisco Javier",86,clean
168,C00700,Armando Linda,7235,"LIWINAG, Armando",86,clean
233,C01927,Mohammed Sommer,6635,"OMAR, Mohammed",86,clean
286,C01889,Ana Taylor,43436,"TAYLOR, Jina",86,clean


In [5]:
alerts.to_csv("../data/processed/alerts.csv", index=False)